# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset Croissant schema is provided at:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}:\n{metadata.description}\n")
print(f"Version: {metadata.version}, Published: {metadata.datePublished}")

## 2. Data Overview
Let's review the available record sets and their fields, referencing each by their `@id` as per the Croissant standard.

In [ ]:
# List all record sets and fields
print("Record sets defined in the Croissant schema:")
for record_set in dataset.record_sets():
    print(f"- RecordSet @id: {record_set['@id']}  Name: {record_set.get('name','(unnamed)')}")
    fields = record_set['field'] if isinstance(record_set['field'], list) else [record_set['field']]
    print("  Fields:")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - Field @id: {field_id}")

To illustrate with an example, let's list the first 3 records from the main record set using its `@id`.

In [ ]:
# Let's assume the principal record set is 'https://api.app.sen.science/frontiers/7160411/83e282e0-a458-4782-b7a1-e9a2bbece0c7'.
# If in doubt, copy its @id from the previous output.
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/83e282e0-a458-4782-b7a1-e9a2bbece0c7'

print(f"First 3 records from record set {main_record_set_id} (by @id):")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
Let's extract all data from the main record set into a Pandas DataFrame for further analysis, referencing record set and field `@id`s.

In [ ]:
# List all record set @ids
record_sets = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set {record_set_id}")

if main_record_set_id in dataframes:
    print("Available columns (fields, referenced by @id):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"Record set {main_record_set_id} not found or empty.")

## 4. Exploratory Data Analysis (EDA)
Let us: 
* Filter based on a numeric field (for example, GAD-7 total score if available), 
* Normalize values,
* Group by another attribute (e.g. gender by its field `@id`), referencing everything by `@id`.

In [ ]:
# Identify example numeric field (by @id), e.g., gad7_total_score
# Replace with correct @id from your data schema. We'll use 'gad7_total' as a placeholder.
numeric_field = 'gad7_total'  # replace with actual field @id (column name)
group_field = 'gender'        # replace with actual field @id for gender (column name)

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold} (total: {len(filtered_df)}):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df)
    else:
        print(f"'{group_field}' not in DataFrame columns.")
else:
    print(f"Field '{numeric_field}' not found in DataFrame. Please check available columns:")
    if df is not None:
        print(df.columns.tolist())

## 5. Visualization
Let us visualize the distribution of a numeric field. All field references use the correct `@id` as column name.

* Histogram of GAD-7 total scores
* Box plot by gender

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field} (referenced by @id)")
        plt.show()
else:
    print(f"Field '{numeric_field}' not found for visualization.")

## 6. Conclusion
- We loaded and explored the FAIR² Mental Health Survey data for Kilifi County, using only entity `@id` values for schema references.
- Using `mlcroissant`, we inspected record sets and fields and extracted data into DataFrames.
- We demonstrated EDA and basic visualizations, referencing fields by schema `@id` throughout.

You can now further extend this notebook to incorporate advanced analysis, cross-record set joins, or custom reporting relevant to your use case.